In [4]:
import pygs
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf
import os
import json
pd.options.mode.chained_assignment = None #don't show warning flags
%matplotlib inline

from IPython.display import display, HTML
display(HTML(
    '<style>'
        '#notebook { padding-top:0px !important; } ' 
        '.container { width:97% !important; } '
        '.end_space { min-height:0px !important; } '
    '</style>'
))

pd.set_option('display.max_columns', None)
sns.set(rc={'figure.figsize':(16.7,4.27)})

In [59]:
### GET DATA ###
def normalize_decklist(tournament_decklist, record_path):
    cleaned_decklist = pd.json_normalize(tournament_decklist['Decks'], meta=['Date', 'Player', 'Result'], record_path = record_path)
    cleaned_decklist['deck_type'] = record_path
    cleaned_decklist['join_field'] = 1
    return cleaned_decklist

def get_tournament_info(decklist_json):
    tournament_info = pd.json_normalize(decklist_json['Tournament'])
    tournament_info.drop(['Uri'],axis = 1, inplace = True)
    tournament_info['join_field'] = 1 
    return tournament_info

def cleanup_decklist_name(deck_data):
    deck_data.rename({'Count':'card_count', 'CardName':'card_name', 'Date':'tournament_date', 
                        'Player':'player_name', 'Result':'result', 'Name' : 'tournament_name'}, axis = 1, inplace = True)
    
def cleanup_decklist_order(deck_data):
    deck_data = deck_data[['card_name', 'card_count', 'deck_type', 'tournament_name', 'result', 'tournament_date', 'player_name']]
    return deck_data

def cleanup_decklist_data(deck_data):
    cleanup_decklist_name(deck_data)
    deck_data.drop(['join_field'],axis=1, inplace=True)
    deck_data = cleanup_decklist_order(deck_data)
    return deck_data

def get_decklist_data(tournament_decklist):
    main_decklist = normalize_decklist(tournament_decklist, 'Mainboard')
    side_decklist = normalize_decklist(tournament_decklist, 'Sideboard')
    overall_decklist = pd.concat([main_decklist, side_decklist])
    overall_decklist.drop(['Date'],axis=1, inplace=True) #gotten from tournament data
    
    tournament_info = get_tournament_info(tournament_decklist)
    overall_decklist = pd.merge(overall_decklist, tournament_info, on=['join_field'], how='inner')
    overall_decklist = cleanup_decklist_data(overall_decklist)
    return overall_decklist

def create_placement(placement_stop):
    keep_tournament_result = ['7-0', '6-0', '5-0','4-0', '6-1', '5-1', '4-1', '5-2', '1st Place', '2nd Place', '3rd Place'] #can't use fuzzy matching as it'll bring things like 18th place in
    placement = []
    for i in range(4, placement_stop + 1):
        keep_tournament_result.append(str(i) + 'th Place')
    return keep_tournament_result
        
def filter_tournament_result(decklist_data):
    keep_tournament_result = create_placement(8)
    decklist_data = decklist_data[decklist_data['result'].isin(keep_tournament_result)]
    return decklist_data

def get_tournament_decklists(mtg_format, directory):
    overall_decklist_data = pd.DataFrame()

    current_path = os.getcwd()
    current_path = current_path + "\\" + directory
    
    for root, dir_names, file_names in os.walk(current_path):
        print('checking results in: {}'.format(root))
        for f in file_names:
            if mtg_format in f:
                print(" * Adding results from: {}".format(f))
                fname = os.path.join(root, f)
                tournament_decklist = open(fname)
                tournament_decklist = json.load(tournament_decklist)
                tournament_decklist = get_decklist_data(tournament_decklist)
                tournament_decklist = filter_tournament_result(tournament_decklist)
                
                overall_decklist_data = pd.concat([overall_decklist_data, tournament_decklist])
        print("")
    return overall_decklist_data

def create_tournament_date():
    tournament_decklists['tournament_date'] = pd.to_datetime(tournament_decklists['tournament_date']).dt.normalize()
    tournament_decklists['tournament_date'] = tournament_decklists['tournament_date'].dt.tz_localize(None)

### DATA ANALYSIS ###

In [ ]:
generate_data = True
csv_file_name = 'modern_tournament_decklists_17_18.csv'

if generate_data:
    tournament_decklists = get_tournament_decklists('modern', 'deck_data') #only care about modern
    tournament_decklists.to_csv(csv_file_name, encoding='utf-8', index=False)
else:
    print("Data already made, skipping writing process")
    tournament_decklists = pd.read_csv(csv_file_name) 
    tournament_decklists.head(5)

 * Adding results from: modern-challenge-top-8-2017-05-2010640187.json

checking results in: G:\My Drive\CS\modern_cube\deck_data\2017\05\21

checking results in: G:\My Drive\CS\modern_cube\deck_data\2017\05\22

checking results in: G:\My Drive\CS\modern_cube\deck_data\2017\05\23

checking results in: G:\My Drive\CS\modern_cube\deck_data\2017\05\24

checking results in: G:\My Drive\CS\modern_cube\deck_data\2017\05\25

checking results in: G:\My Drive\CS\modern_cube\deck_data\2017\05\26

checking results in: G:\My Drive\CS\modern_cube\deck_data\2017\05\27
 * Adding results from: modern-challenge-2017-05-2710648056.json

checking results in: G:\My Drive\CS\modern_cube\deck_data\2017\05\28

checking results in: G:\My Drive\CS\modern_cube\deck_data\2017\05\29

checking results in: G:\My Drive\CS\modern_cube\deck_data\2017\05\30

checking results in: G:\My Drive\CS\modern_cube\deck_data\2017\05\31

checking results in: G:\My Drive\CS\modern_cube\deck_data\2017\06

checking results in: G:\My


checking results in: G:\My Drive\CS\modern_cube\deck_data\2017\09\11

checking results in: G:\My Drive\CS\modern_cube\deck_data\2017\09\13

checking results in: G:\My Drive\CS\modern_cube\deck_data\2017\09\12

checking results in: G:\My Drive\CS\modern_cube\deck_data\2017\09\15

checking results in: G:\My Drive\CS\modern_cube\deck_data\2017\09\14

checking results in: G:\My Drive\CS\modern_cube\deck_data\2017\09\16
 * Adding results from: modern-challenge-2017-09-1610878902.json

checking results in: G:\My Drive\CS\modern_cube\deck_data\2017\09\18

checking results in: G:\My Drive\CS\modern_cube\deck_data\2017\09\20

checking results in: G:\My Drive\CS\modern_cube\deck_data\2017\09\17

checking results in: G:\My Drive\CS\modern_cube\deck_data\2017\09\19

checking results in: G:\My Drive\CS\modern_cube\deck_data\2017\09\21

checking results in: G:\My Drive\CS\modern_cube\deck_data\2017\09\22

checking results in: G:\My Drive\CS\modern_cube\deck_data\2017\09\24

checking results in: G:\


checking results in: G:\My Drive\CS\modern_cube\deck_data\2018

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\01

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\01\04

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\01\07

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\01\02

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\01\03

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\01\01

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\01\10

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\01\09

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\01\08

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\01\11

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\01\13
 * Adding results from: modern-challenge-2018-01-1311104098.json

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\01\05

checking results in: G:\My Drive\


checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\02\22
 * Adding results from: modern-league-2018-02-222868.json
 * Adding results from: modern-league-2018-02-222844.json

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\02\23
 * Adding results from: modern-league-2018-02-232868.json
 * Adding results from: modern-league-2018-02-232844.json

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\02\24
 * Adding results from: modern-league-2018-02-242844.json
 * Adding results from: modern-league-2018-02-242868.json
 * Adding results from: modern-challenge-2018-02-2411192645.json

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\02\25
 * Adding results from: modern-league-2018-02-252868.json
 * Adding results from: modern-league-2018-02-252844.json

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\02\26
 * Adding results from: modern-league-2018-02-262844.json
 * Adding results from: modern-league-2018-02-262868.json

chec


checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\04\04
 * Adding results from: modern-league-2018-04-042868.json
 * Adding results from: modern-league-2018-04-042844.json

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\04\05
 * Adding results from: modern-league-2018-04-052868.json
 * Adding results from: modern-league-2018-04-052844.json

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\04\06
 * Adding results from: modern-league-2018-04-062868.json
 * Adding results from: modern-league-2018-04-062844.json

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\04\07
 * Adding results from: modern-league-2018-04-072868.json
 * Adding results from: modern-league-2018-04-072844.json
 * Adding results from: modern-challenge-2018-04-0711291058.json

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\04\08
 * Adding results from: modern-league-2018-04-082868.json
 * Adding results from: modern-league-2018-04-082844.json

chec

...


checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\06\27
 * Adding results from: modern-league-2018-06-273096.json
 * Adding results from: modern-league-2018-06-273136.json

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\06\28
 * Adding results from: modern-league-2018-06-283136.json
 * Adding results from: modern-league-2018-06-283096.json

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\06\29
 * Adding results from: modern-league-2018-06-293096.json
 * Adding results from: modern-league-2018-06-293136.json

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\06\30
 * Adding results from: modern-league-2018-06-303096.json
 * Adding results from: modern-league-2018-06-303136.json
 * Adding results from: modern-challenge-2018-06-3011460907.json

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\07

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\07\01
 * Adding results from: modern-league-2018-07-013096.j


checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\08\08
 * Adding results from: modern-league-2018-08-083306.json
 * Adding results from: modern-league-2018-08-083346.json

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\08\09
 * Adding results from: modern-league-2018-08-093346.json
 * Adding results from: modern-league-2018-08-093306.json

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\08\10
 * Adding results from: modern-league-2018-08-103346.json
 * Adding results from: modern-league-2018-08-103306.json

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\08\11
 * Adding results from: modern-league-2018-08-113346.json
 * Adding results from: modern-league-2018-08-113306.json
 * Adding results from: modern-challenge-2018-08-1111542642.json

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\08\12
 * Adding results from: modern-league-2018-08-123346.json
 * Adding results from: modern-league-2018-08-123306.json

chec


checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\09\19
 * Adding results from: modern-league-2018-09-193306.json
 * Adding results from: modern-league-2018-09-193346.json

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\09\20
 * Adding results from: modern-league-2018-09-203346.json
 * Adding results from: modern-league-2018-09-203306.json

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\09\21
 * Adding results from: modern-league-2018-09-213306.json
 * Adding results from: modern-league-2018-09-213346.json

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\09\22
 * Adding results from: modern-league-2018-09-223346.json
 * Adding results from: modern-challenge-2018-09-2211615687.json
 * Adding results from: modern-league-2018-09-223306.json

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\09\23
 * Adding results from: modern-league-2018-09-233346.json
 * Adding results from: modern-league-2018-09-233306.json

chec


checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\10\31
 * Adding results from: modern-league-2018-10-313552.json
 * Adding results from: modern-league-2018-10-313520.json

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\11

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\11\01
 * Adding results from: modern-league-2018-11-013520.json
 * Adding results from: modern-league-2018-11-013552.json

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\11\02
 * Adding results from: modern-league-2018-11-023552.json
 * Adding results from: modern-league-2018-11-023520.json

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\11\03
 * Adding results from: modern-league-2018-11-033552.json
 * Adding results from: modern-challenge-2018-11-0311681656.json
 * Adding results from: modern-league-2018-11-033520.json

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\11\04
 * Adding results from: modern-league-2018-11-043552.j


checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\12\12
 * Adding results from: modern-league-2018-12-123552.json
 * Adding results from: modern-league-2018-12-123520.json

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\12\13
 * Adding results from: modern-league-2018-12-133520.json
 * Adding results from: modern-league-2018-12-133552.json

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\12\14
 * Adding results from: modern-league-2018-12-143520.json
 * Adding results from: modern-league-2018-12-143552.json

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\12\15
 * Adding results from: modern-league-2018-12-153520.json
 * Adding results from: modern-challenge-2018-12-1511740707.json
 * Adding results from: modern-league-2018-12-153552.json

checking results in: G:\My Drive\CS\modern_cube\deck_data\2018\12\16
 * Adding results from: modern-league-2018-12-163520.json
 * Adding results from: modern-league-2018-12-163552.json

chec

In [61]:
tournament_decklists = pd.read_csv(csv_file_name)


create_tournament_date()
tournament_decklists.head(5)

,card_name,card_count,deck_type,tournament_name,result,tournament_date,player_name
0,Blood Crypt,1,Mainboard,Modern PTQ Preliminary,5-0,2017-01-01,__Ghost__
1,Bloodstained Mire,1,Mainboard,Modern PTQ Preliminary,5-0,2017-01-01,__Ghost__
2,Delver of Secrets,4,Mainboard,Modern PTQ Preliminary,5-0,2017-01-01,__Ghost__
3,Gitaxian Probe,3,Mainboard,Modern PTQ Preliminary,5-0,2017-01-01,__Ghost__
4,Island,2,Mainboard,Modern PTQ Preliminary,5-0,2017-01-01,__Ghost__


In [57]:
tournament_decklists.dtypes

card_name                  object
card_count                  int64
deck_type                  object
tournament_name            object
result                     object
tournament_date    datetime64[ns]
player_name                object
dtype: object

In [62]:
### SCRYFALL DATA ###


,object,id,oracle_id,multiverse_ids,mtgo_id,mtgo_foil_id,tcgplayer_id,cardmarket_id,name,lang,released_at,uri,scryfall_uri,layout,highres_image,image_status,mana_cost,cmc,type_line,oracle_text,colors,color_identity,keywords,games,reserved,foil,nonfoil,finishes,oversized,promo,reprint,variation,set_id,set,set_name,set_type,set_uri,set_search_uri,scryfall_set_uri,rulings_uri,prints_search_uri,collector_number,digital,rarity,flavor_text,card_back_id,artist,artist_ids,illustration_id,border_color,frame,full_art,textless,booster,story_spotlight,edhrec_rank,image_uris.small,image_uris.normal,image_uris.large,image_uris.png,image_uris.art_crop,image_uris.border_crop,legalities.standard,legalities.future,legalities.historic,legalities.timeless,legalities.gladiator,legalities.pioneer,legalities.explorer,legalities.modern,legalities.legacy,legalities.pauper,legalities.vintage,legalities.penny,legalities.commander,legalities.oathbreaker,legalities.standardbrawl,legalities.brawl,legalities.alchemy,legalities.paupercommander,legalities.duel,legalities.oldschool,legalities.premodern,legalities.predh,prices.usd,prices.usd_foil,prices.usd_etched,prices.eur,prices.eur_foil,prices.tix,related_uris.gatherer,related_uris.tcgplayer_infinite_articles,related_uris.tcgplayer_infinite_decks,related_uris.edhrec,purchase_uris.tcgplayer,purchase_uris.cardmarket,purchase_uris.cardhoarder,security_stamp,preview.source,preview.source_uri,preview.previewed_at,power,toughness,penny_rank,arena_id,promo_types,all_parts,frame_effects,produced_mana,watermark,card_faces,tcgplayer_etched_id,loyalty,life_modifier,hand_modifier,attraction_lights,color_indicator,content_warning
0,card,86bf43b1-8d4e-4759-bb2d-0b2e03ba7012,0004ebd0-dfd6-4276-b4a6-de0003e94237,[15862],15870.0,15871.0,3094.0,3081.0,Static Orb,en,2001-04-11,https://api.scryfall.com/cards/86bf43b1-8d4e-4...,https://scryfall.com/card/7ed/319/static-orb?u...,normal,True,highres_scan,{3},3.0,Artifact,"As long as Static Orb is untapped, players can...",[],[],[],"[paper, mtgo]",False,False,True,[nonfoil],False,False,True,False,230f38aa-9511-4db8-a3aa-aeddbc3f7bb9,7ed,Seventh Edition,core,https://api.scryfall.com/sets/230f38aa-9511-4d...,https://api.scryfall.com/cards/search?order=se...,https://scryfall.com/sets/7ed?utm_source=api,https://api.scryfall.com/cards/86bf43b1-8d4e-4...,https://api.scryfall.com/cards/search?order=re...,319,False,rare,The warriors fought against the paralyzing wav...,0aeebaf5-8c7d-4636-9e82-8c27447861f7,Terese Nielsen,[eb55171c-2342-45f4-a503-2d5a75baf752],6f8b3b2c-252f-4f95-b621-712c82be38b5,white,1997,False,False,True,False,3863.0,https://cards.scryfall.io/small/front/8/6/86bf...,https://cards.scryfall.io/normal/front/8/6/86b...,https://cards.scryfall.io/large/front/8/6/86bf...,https://cards.scryfall.io/png/front/8/6/86bf43...,https://cards.scryfall.io/art_crop/front/8/6/8...,https://cards.scryfall.io/border_crop/front/8/...,not_legal,not_legal,not_legal,not_legal,not_legal,not_legal,not_legal,not_legal,legal,not_legal,legal,not_legal,legal,legal,not_legal,not_legal,not_legal,not_legal,legal,not_legal,legal,legal,19.41,None,None,9.42,None,0.24,https://gatherer.wizards.com/Pages/Card/Detail...,https://tcgplayer.pxf.io/c/4931599/1830156/210...,https://tcgplayer.pxf.io/c/4931599/1830156/210...,https://edhrec.com/route/?cc=Static+Orb,https://tcgplayer.pxf.io/c/4931599/1830156/210...,https://www.cardmarket.com/en/Magic/Products/S...,https://www.cardhoarder.com/cards/15870?affili...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,card,7050735c-b232-47a6-a342-01795bfd0d46,0006faf6-7a61-426c-9034-579f2cfcfa83,[370780],49283.0,49284.0,69965.0,262945.0,Sensory Deprivation,en,2013-07-19,https://api.scryfall.com/cards/7050735c-b232-4...,https://scryfall.com/card/m14/71/sensory-depri...,normal,True,highres_scan,{U},1.0,Enchantment — Aura,Enchant creature\nEnchanted creature gets -3/-0.,[U],[U],[Enchant],"[paper, mtgo]",False,True,True,"[nonfoil, foil]",False,Fa

In [63]:
csv_file_name = 'scryfall.csv'

if generate_data:
    #data via https://scryfall.com/docs/api/bulk-data; bulk data used to prevent too many API calls
    scryfall = open('scryfall.json', encoding="utf8")
    scryfall = json.load(scryfall)
    scryfall = pd.json_normalize(scryfall)
    scryfall.to_csv(csv_file_name, encoding='utf-8', index=False)
else:
    print("Data already made, skipping writing process")
    scryfall = pd.read_csv(csv_file_name) 
    scryfall.head(5)

In [64]:
scryfall

,object,id,oracle_id,multiverse_ids,mtgo_id,mtgo_foil_id,tcgplayer_id,cardmarket_id,name,lang,released_at,uri,scryfall_uri,layout,highres_image,image_status,mana_cost,cmc,type_line,oracle_text,colors,color_identity,keywords,games,reserved,foil,nonfoil,finishes,oversized,promo,reprint,variation,set_id,set,set_name,set_type,set_uri,set_search_uri,scryfall_set_uri,rulings_uri,prints_search_uri,collector_number,digital,rarity,flavor_text,card_back_id,artist,artist_ids,illustration_id,border_color,frame,full_art,textless,booster,story_spotlight,edhrec_rank,image_uris.small,image_uris.normal,image_uris.large,image_uris.png,image_uris.art_crop,image_uris.border_crop,legalities.standard,legalities.future,legalities.historic,legalities.timeless,legalities.gladiator,legalities.pioneer,legalities.explorer,legalities.modern,legalities.legacy,legalities.pauper,legalities.vintage,legalities.penny,legalities.commander,legalities.oathbreaker,legalities.standardbrawl,legalities.brawl,legalities.alchemy,legalities.paupercommander,legalities.duel,legalities.oldschool,legalities.premodern,legalities.predh,prices.usd,prices.usd_foil,prices.usd_etched,prices.eur,prices.eur_foil,prices.tix,related_uris.gatherer,related_uris.tcgplayer_infinite_articles,related_uris.tcgplayer_infinite_decks,related_uris.edhrec,purchase_uris.tcgplayer,purchase_uris.cardmarket,purchase_uris.cardhoarder,security_stamp,preview.source,preview.source_uri,preview.previewed_at,power,toughness,penny_rank,arena_id,promo_types,all_parts,frame_effects,produced_mana,watermark,card_faces,tcgplayer_etched_id,loyalty,life_modifier,hand_modifier,attraction_lights,color_indicator,content_warning
0,card,86bf43b1-8d4e-4759-bb2d-0b2e03ba7012,0004ebd0-dfd6-4276-b4a6-de0003e94237,[15862],15870.0,15871.0,3094.0,3081.0,Static Orb,en,2001-04-11,https://api.scryfall.com/cards/86bf43b1-8d4e-4...,https://scryfall.com/card/7ed/319/static-orb?u...,normal,True,highres_scan,{3},3.0,Artifact,"As long as Static Orb is untapped, players can...",[],[],[],"[paper, mtgo]",False,False,True,[nonfoil],False,False,True,False,230f38aa-9511-4db8-a3aa-aeddbc3f7bb9,7ed,Seventh Edition,core,https://api.scryfall.com/sets/230f38aa-9511-4d...,https://api.scryfall.com/cards/search?order=se...,https://scryfall.com/sets/7ed?utm_source=api,https://api.scryfall.com/cards/86bf43b1-8d4e-4...,https://api.scryfall.com/cards/search?order=re...,319,False,rare,The warriors fought against the paralyzing wav...,0aeebaf5-8c7d-4636-9e82-8c27447861f7,Terese Nielsen,[eb55171c-2342-45f4-a503-2d5a75baf752],6f8b3b2c-252f-4f95-b621-712c82be38b5,white,1997,False,False,True,False,3863.0,https://cards.scryfall.io/small/front/8/6/86bf...,https://cards.scryfall.io/normal/front/8/6/86b...,https://cards.scryfall.io/large/front/8/6/86bf...,https://cards.scryfall.io/png/front/8/6/86bf43...,https://cards.scryfall.io/art_crop/front/8/6/8...,https://cards.scryfall.io/border_crop/front/8/...,not_legal,not_legal,not_legal,not_legal,not_legal,not_legal,not_legal,not_legal,legal,not_legal,legal,not_legal,legal,legal,not_legal,not_legal,not_legal,not_legal,legal,not_legal,legal,legal,19.41,None,None,9.42,None,0.24,https://gatherer.wizards.com/Pages/Card/Detail...,https://tcgplayer.pxf.io/c/4931599/1830156/210...,https://tcgplayer.pxf.io/c/4931599/1830156/210...,https://edhrec.com/route/?cc=Static+Orb,https://tcgplayer.pxf.io/c/4931599/1830156/210...,https://www.cardmarket.com/en/Magic/Products/S...,https://www.cardhoarder.com/cards/15870?affili...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,card,7050735c-b232-47a6-a342-01795bfd0d46,0006faf6-7a61-426c-9034-579f2cfcfa83,[370780],49283.0,49284.0,69965.0,262945.0,Sensory Deprivation,en,2013-07-19,https://api.scryfall.com/cards/7050735c-b232-4...,https://scryfall.com/card/m14/71/sensory-depri...,normal,True,highres_scan,{U},1.0,Enchantment — Aura,Enchant creature\nEnchanted creature gets -3/-0.,[U],[U],[Enchant],"[paper, mtgo]",False,True,True,"[nonfoil, foil]",False,Fa

In [ ]:
#hardcode the ban list (this has cards post 2018 in case we ever change what time period we look at)
#any card
["Ancient Den", "Arcum's Astrolabe", "Birthing Pod", "Blazing Shoal", "Bridge From Below", "Chrome Mox", "Cloudpost", "Dark Depths", "Deathrite Shaman", "Dig Through Time", "Dread Return", "Eye of Ugin", "Faithless Looting", "Field of the Dead", "Fury", "Gitaxian Probe", "Glimpse of Nature", "Golgari Grave-Troll", "Great Furnace", "Green Sun's Zenith", "Hogaak, Arisen Necropolis", "Hypergenesis", "Krark-Clan Ironworks", "Lurrus of the Dream-Den", "Mental Misstep", "Mox Opal", "Mycosynth Lattice", "Mystic Sanctuary", "Oko, Thief of Crowns", "Once Upon a Time", "Ponder", "Punishing Fire", "Rite of Flame", "Seat of the Synod", "Second Sunrise", "Seething Song", "Sensei's Divining Top", "Simian Spirit Guide", "Skullclamp", "Splinter Twin", "Summer Bloom", "Tibalt's Trickery", "Treasure Cruise", "Tree of Tales", "Umezawa's Jitte", "Up the Beanstalk", "Uro, Titan of Nature's Wrath", "Vault of Whispers", "Violent Outburst", "Yorion, Sky Nomad"]

In [ ]:
scryfall.loc[(scryfall['lang'] == 'en') & (scryfall['released_at'] <= '2019-01-01') & (scryfall['name'] == 'Storm Crow')]

In [ ]:
scryfall['lang'].unique()

In [ ]:
pd.json_normalize(scryfall)

In [ ]:
### Weighing System for Cards ###

In [ ]:
# type of tournament
# number of copies
# main vs side deck
# was the card eventually banned
# length of time card was legal in the format (so new cards)
# card type (empthasis on lands)
# mana value
# color

In [ ]:
tournament_weight = tournament_decklists['tournament_name'].value_counts().reset_index(name='count')
tournament_weight.rename({'index':'tournament_name'}, axis = 1, inplace = True)

tournament_weight['total_percent'] = round(tournament_weight['count'] / len(tournament_decklists),4)
tournament_weight